# UK 06 · Coverage audit and leaf hash gate

Execute in numbered order. DONE units are skipped with a reason; closed results are read-only and only receive views under `results/_views/`.


In [ ]:
import os
from pathlib import Path
from IPython.display import display
from sglib.core.infra.paths import find_repo_root
from sglib.dataoverview.handoff import load_bundle as load_dataoverview
from sglib.generator import downstream as generator_downstream
from sglib.experiment import stage

UPSTREAM = {"data": load_dataoverview, "generator": generator_downstream.load_handoff}
from sglib.experiment.overview import tables

REPO = find_repo_root(Path.cwd())
COUNTRY = "uk"
PROFILE = os.environ.get("SG_PROFILE", "formal")
RESULTS_ROOT = os.environ.get("SG_RESULTS_ROOT")
OPTIONS = dict(profile=PROFILE, results_root=RESULTS_ROOT, upstream=UPSTREAM)
ctx = stage.country_context(REPO, COUNTRY, **OPTIONS)
print(f"results_root={ctx.results} read_only={ctx.read_only}")


In [ ]:
stage.run_step(REPO, COUNTRY, ['audit'], **OPTIONS)


## Coverage audit and leaf hash gate

`coverage` reads every unit's receipt or marker and reports expected versus produced coordinates. The `audit`
unit writes `audit.json` once all units are DONE (on a closed root it is generated into `results/_views/` and
placed after confirmation). `run_gate` then compares the leaf manifest with the files on disk: leaf = unit;
a FAIL names the leaf and the files; a root without a manifest only generates one.


In [ ]:
from sglib.experiment.manifest import run_gate, write_setup
display(tables.coverage(ctx))
write_setup(ctx)
report = run_gate(ctx)
if report["status"] == "FAIL":
    raise SystemExit(1)
if report["status"] == "PASS" and (ctx.root / "audit.json").is_file():
    display(tables.audit(ctx))
